<div dir="rtl">

# ۱ – آماده‌سازی داده (Persian News Dataset)

در این نوت‌بوک، دیتاست خبری فارسی را از پوشه‌ی `data/raw` می‌خوانید، یک زیرمجموعه‌ی کوچک و قابل‌مدیریت از آن می‌سازید، داده‌ها را کمی تمیز می‌کنید و نتیجه را در مسیر `data/processed/news_subset.csv` ذخیره می‌کنید.

        https://www.kaggle.com/datasets/amirzenoozi/persian-news-dataset/

</div>


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path 

In [ ]:
main_dir = "Desktop/BOOTCAMP/S9-RAG/پروژه  دوره RAG/rag-persian-news-lite-main"

raw_path = Path(f'{main_dir}/data/raw').mkdir(parents=True, exist_ok=True)
subset_path = Path(f'{main_dir}../data/processed').mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv('/kaggle/input/persion-news-dataset/news.csv')

print(f"Dataset shape: {df.shape}")
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
print("\n\nMissing values:")
print(df.isnull().sum())
print("\n\nCategories:")
print(df['category'].value_counts())
def clean_persian_text(text):
    """Clean Persian text with basic normalization"""
    if pd.isna(text):
        return ""
    
    text = str(text)
     
    text = text.replace('ك', 'ک').replace('ي', 'ی')
    text = text.replace('ٔ', '').replace('ء', '')
     
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)۰-۹]', '', text, flags=re.UNICODE)
    
    return text.strip()
 
df['title_clean'] = df['title'].apply(clean_persian_text)
df['description_clean'] = df['description'].apply(clean_persian_text)
 
df = df[(df['title_clean'] != "") | (df['description_clean'] != "")]
 
df['combined_text'] = df['title_clean'] + " " + df['description_clean']
 
df = df[df['combined_text'].str.len() >= 50]

print(f"\nAfter cleaning: {df.shape}")


In [ ]:
subset_size = 15000

if len(df) > subset_size: 
    df_subset = df.groupby('category', group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(subset_size * len(x) / len(df))), random_state=42)
    ).reset_index(drop=True)
     
    if len(df_subset) < subset_size:
        remaining = subset_size - len(df_subset)
        additional = df[~df.index.isin(df_subset.index)].sample(remaining, random_state=42)
        df_subset = pd.concat([df_subset, additional]).reset_index(drop=True)
else:
    df_subset = df.copy()

print(f"\nSubset size: {len(df_subset)}") 
print(df_subset['category'].value_counts())


In [ ]:
 
df_final = df_subset[['title_clean', 'description_clean', 'combined_text', 'category', 'date']].copy()
df_final.columns = ['title', 'description', 'text', 'category', 'date']
 
df_final['doc_id'] = [f'doc_{i:05d}' for i in range(len(df_final))]
 
df_final = df_final[['doc_id', 'title', 'description', 'text', 'category', 'date']]
 

In [ ]:
output_path = f'{main_dir}/data/processed/news_subset.csv'
df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n✅ Data saved to: {output_path}")
print("\nFinal dataset info:")
print(df_final.info())
print("\nSample rows:")
df_final.head()